# 🏥 Insurance Cost Prediction
## Machine Learning Project — Applied AI Bootcamp

**الهدف:** بناء موديل ML يتنبأ بتكلفة التأمين الطبي السنوية لشخص بناءً على معلوماته الشخصية والصحية.

**نوع المشكلة:** Regression (تنبؤ بقيمة عددية مستمرة — التكلفة بالدولار)

## 1. تحميل المكتبات والداتا

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

sns.set_style('whitegrid')

df = pd.read_csv('insurance.csv')
print('Shape:', df.shape)
df.head()

## 2. استكشاف الداتا (EDA)

**الأعمدة:**
- `age` — عمر الشخص
- `sex` — الجنس (male/female)
- `bmi` — مؤشر كتلة الجسم
- `children` — عدد الأطفال المؤمّن عليهم
- `smoker` — مدخن أو لا (yes/no)
- `region` — منطقة السكن
- `charges` — **الهدف**: التكلفة السنوية بالدولار

In [ ]:
df.info()
print('\nالقيم المفقودة:')
print(df.isnull().sum())
df.describe()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0,0].hist(df['charges'], bins=40, color='steelblue', edgecolor='black')
axes[0,0].set_title('Distribution of Charges')
axes[0,0].set_xlabel('Charges ($)')

axes[0,1].scatter(df['age'], df['charges'], alpha=0.4, c='coral')
axes[0,1].set_title('Age vs Charges')

colors = df['smoker'].map({'yes':'red', 'no':'green'})
axes[0,2].scatter(df['bmi'], df['charges'], c=colors, alpha=0.5)
axes[0,2].set_title('BMI vs Charges (Red=Smoker)')

sns.boxplot(data=df, x='smoker', y='charges', ax=axes[1,0])
axes[1,0].set_title('Smoker Effect')

sns.boxplot(data=df, x='children', y='charges', ax=axes[1,1])
axes[1,1].set_title('Children vs Charges')

sns.boxplot(data=df, x='region', y='charges', ax=axes[1,2])
axes[1,2].set_title('Region vs Charges')
axes[1,2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

**ملاحظات من الرسومات:**
1. توزيع التكلفة منحرف لليمين — معظم الناس بأرقام منخفضة، وأقلية بتكاليف عالية جداً
2. التدخين له تأثير ضخم — المدخنين يدفعون أضعاف غير المدخنين
3. العمر علاقته خطية تقريباً مع التكلفة
4. الـ BMI العالي مع التدخين = تكلفة جداً مرتفعة

## 3. تجهيز الداتا للتدريب

In [ ]:
X = df.drop('charges', axis=1)
y = df['charges']

categorical = ['sex', 'smoker', 'region']
numeric = ['age', 'bmi', 'children']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), categorical)
], remainder='passthrough')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 4. تدريب 3 موديلات والمقارنة

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, random_state=42)
}

results = {}
for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    
    results[name] = {
        'pipeline': pipe,
        'MAE':  mean_absolute_error(y_test, preds),
        'RMSE': np.sqrt(mean_squared_error(y_test, preds)),
        'R2':   r2_score(y_test, preds)
    }

comparison = pd.DataFrame({k: {m: v[m] for m in ['MAE','RMSE','R2']} for k,v in results.items()}).T
comparison

## 5. اختيار الموديل الأفضل + Feature Importance

In [ ]:
best_name = max(results, key=lambda k: results[k]['R2'])
best_pipe = results[best_name]['pipeline']
print(f'🏆 الموديل الأفضل: {best_name}')
print(f'R² = {results[best_name]["R2"]:.4f}')
print(f'MAE = ${results[best_name]["MAE"]:,.2f}')

# Feature importance
feature_names = (list(best_pipe.named_steps['prep'].named_transformers_['cat']
                      .get_feature_names_out(categorical)) + numeric)
importances = best_pipe.named_steps['model'].feature_importances_

fig, ax = plt.subplots(figsize=(10, 6))
idx = np.argsort(importances)
ax.barh([feature_names[i] for i in idx], importances[idx], color='teal')
ax.set_title(f'Feature Importance — {best_name}')
plt.tight_layout()
plt.show()

In [ ]:
preds = best_pipe.predict(X_test)
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, preds, alpha=0.5, color='purple')
lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
ax.plot(lims, lims, 'r--', label='Perfect Prediction')
ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
ax.set_title(f'{best_name}: Predicted vs Actual')
ax.legend(); plt.show()

## 6. حفظ الموديل + أمثلة تنبؤ

In [ ]:
joblib.dump(best_pipe, 'insurance_model.pkl')

# تنبؤ على شخص جديد
person = pd.DataFrame([{
    'age': 35, 'sex': 'male', 'bmi': 28.5,
    'children': 2, 'smoker': 'no', 'region': 'southeast'
}])
print(f'تكلفة شخص غير مدخن: ${best_pipe.predict(person)[0]:,.2f}')

person['smoker'] = 'yes'
print(f'تكلفة نفس الشخص لو مدخن: ${best_pipe.predict(person)[0]:,.2f}')

## 7. (اختياري) تدريب بـ AutoGluon

AutoGluon يجرّب عشرات الموديلات تلقائياً ويعمل ensemble لها. شغّل الكود التالي على جهازك بعد:
```bash
pip install autogluon.tabular
```

In [ ]:
# from autogluon.tabular import TabularPredictor
# 
# train_data = df.sample(frac=0.8, random_state=42)
# test_data  = df.drop(train_data.index)
# 
# predictor = TabularPredictor(
#     label='charges',
#     problem_type='regression',
#     eval_metric='r2'
# ).fit(train_data, time_limit=300, presets='best_quality')
# 
# predictor.leaderboard(test_data, silent=True)